In [4]:
# C.1 Setup do SparkSession

from pyspark.sql import SparkSession
spark = (
SparkSession.builder
.appName("pipeline-sensores-iot")
.master("local[*]")
.config("spark.sql.files.maxPartitionBytes", "128mb")
.config("spark.sql.adaptive.enabled", "true")
.config("spark.sql.shuffle.partitions", "200")
.config("spark.driver.memory", "2g")
.config("spark.executor.memory", "4g")
.getOrCreate()
)

In [5]:
# C.2 Leitura do CSV com schema explícito

from pyspark.sql.types import (
StructType, StructField, IntegerType, StringType,
TimestampType, DoubleType
)
schema = StructType([
StructField("id_leitura", IntegerType(), True),
StructField("timestamp", TimestampType(), True),
StructField("id_sensor", StringType(), True),
StructField("estacao", StringType(), True),
StructField("temperatura", DoubleType(), True),
StructField("umidade", DoubleType(), True),
StructField("bateria", DoubleType(), True),
])
df = (
spark.read
.option("header", True)
.schema(schema)
.csv("leituras_sensores_100k.csv")
)

In [6]:
# C.3 Operações equivalentes às de A.3

from pyspark.sql import functions as F

# 3.1 Contagens
total_leituras = df.count()
print(f"Numero total de leituras: {total_leituras}")

leituras_por_estacao = (
df.groupBy("estacao")
.agg(F.count("*").alias("contagem"))
.orderBy("estacao")
)
leituras_por_estacao.show()

dict_estacao_contagem = {row["estacao"]: row["contagem"] for row in leituras_por_estacao.collect()}
print(dict_estacao_contagem)

leituras_por_sensor = (
df.groupBy("id_sensor")
.agg(F.count("*").alias("contagem"))
.orderBy("id_sensor")
)
leituras_por_sensor.show()

# 3.2 Operacoes matematicas / agregacoes

agg_estacao = (
df.groupBy("estacao")
.agg(
F.avg("temperatura").alias("temperatura_media"),
F.min("temperatura").alias("temperatura_min"),
F.max("temperatura").alias("temperatura_max"),
F.avg("umidade").alias("umidade_media"),
F.min("umidade").alias("umidade_min"),
F.max("umidade").alias("umidade_max"),
F.avg("bateria").alias("bateria_media"),
F.min("bateria").alias("bateria_min"),
F.max("bateria").alias("bateria_max"),
)
.orderBy("estacao")
)
agg_estacao.show()

agg_sensor = (
df.groupBy("id_sensor")
.agg(
F.avg("temperatura").alias("temperatura_media"),
F.min("temperatura").alias("temperatura_min"),
F.max("temperatura").alias("temperatura_max"),
F.avg("bateria").alias("bateria_media"),
F.min("bateria").alias("bateria_min"),
F.max("bateria").alias("bateria_max"),
)
.orderBy("id_sensor")
)
agg_sensor.show()

variacao_sensor = (
df.groupBy("id_sensor")
.agg(
(F.max("bateria") - F.min("bateria")).alias("variacao_bateria"),
(F.max("temperatura") - F.min("temperatura")).alias("variacao_temperatura"),
)
.orderBy("id_sensor")
)
variacao_sensor.show()

LIMIAR_BATERIA = 3.3

percentual_criticas_estacao = (
df.groupBy("estacao")
.agg(
F.count("*").alias("total"),
F.sum(F.when(F.col("bateria") < LIMIAR_BATERIA, 1).otherwise(0)).alias("total_criticas"),
)
.withColumn("percentual_criticas", F.round((F.col("total_criticas") / F.col("total")) * 100, 2))
.orderBy("estacao")
)
percentual_criticas_estacao.show()

# 3.3 Adicao de colunas

df_processado = df.withColumn(
"indice_conforto",
F.round(F.col("temperatura") - (F.col("umidade") / 10), 2),
)

df_processado = df_processado.withColumn(
"status",
F.when(F.col("bateria") < 3.3, "critica")
.when(F.col("bateria") < 3.6, "baixa")
.otherwise("normal"),
)

df_processado.select("id_leitura", "estacao", "id_sensor", "temperatura", "umidade", "bateria", "indice_conforto", "status").show(10)

(
df_processado
.coalesce(1)
.write
.mode("overwrite")
.option("header", True)
.csv("leituras_processadas.csv")
)

Numero total de leituras: 100000
+-------+--------+
|estacao|contagem|
+-------+--------+
|    100|    1416|
|    101|    1492|
|    102|    1356|
|    103|    1381|
|    104|    1460|
|    105|    1399|
|    106|    1481|
|    107|    1374|
|    108|    1390|
|    109|    1424|
|    110|    1399|
|    111|    1410|
|    112|    1372|
|    113|    1384|
|    114|    1416|
|    115|    1364|
|    116|    1420|
|    117|    1389|
|    118|    1352|
|    119|    1431|
+-------+--------+
only showing top 20 rows
{'100': 1416, '101': 1492, '102': 1356, '103': 1381, '104': 1460, '105': 1399, '106': 1481, '107': 1374, '108': 1390, '109': 1424, '110': 1399, '111': 1410, '112': 1372, '113': 1384, '114': 1416, '115': 1364, '116': 1420, '117': 1389, '118': 1352, '119': 1431, '120': 1360, '121': 1432, '122': 1383, '123': 1423, '124': 1403, '125': 1334, '126': 1415, '127': 1390, '128': 1386, '129': 1377, '130': 1485, '60': 1471, '61': 1475, '62': 1431, '63': 1399, '64': 1377, '65': 1381, '66': 1480

In [7]:
# C.4 Medicao de memoria e tempo

import time
import tracemalloc

# tracemalloc mede apenas alocacoes no processo Python do driver.
# O tempo e medido em torno do pipeline completo (leitura + transformacoes + acao final),
# ja que o Spark e lazy e so executa de fato quando ha uma acao.

tracemalloc.start()
inicio = time.time()

df_c4 = (
spark.read
.option("header", True)
.schema(schema)
.csv("leituras_sensores_100k.csv")
)

df_c4_processado = (
df_c4
.withColumn(
"indice_conforto",
F.round(F.col("temperatura") - (F.col("umidade") / 10), 2),
)
.withColumn(
"status",
F.when(F.col("bateria") < 3.3, "critica")
.when(F.col("bateria") < 3.6, "baixa")
.otherwise("normal"),
)
)

total_linhas = df_c4_processado.count()

(
df_c4_processado
.coalesce(1)
.write
.mode("overwrite")
.option("header", True)
.csv("leituras_processadas.csv")
)

fim = time.time()
memoria_atual, memoria_pico = tracemalloc.get_traced_memory()
tracemalloc.stop()

tempo_total = fim - inicio

print(f"Tempo total do pipeline (leitura + transformacoes + escrita): {tempo_total:.2f} segundos")
print(f"Numero de linhas processadas: {total_linhas}")
print(f"Memoria do driver (Python) - atual: {memoria_atual / 1024:.2f} KB")
print(f"Memoria do driver (Python) - pico: {memoria_pico / 1024:.2f} KB")
print("Obs: essa medicao de memoria reflete apenas o processo Python do driver.")
print("O processamento pesado ocorre nos executores JVM e nao e capturado pelo tracemalloc.")

Tempo total do pipeline (leitura + transformacoes + escrita): 4.66 segundos
Numero de linhas processadas: 100000
Memoria do driver (Python) - atual: 38269.25 KB
Memoria do driver (Python) - pico: 42284.72 KB
Obs: essa medicao de memoria reflete apenas o processo Python do driver.
O processamento pesado ocorre nos executores JVM e nao e capturado pelo tracemalloc.
